<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/Lab2_AutogenMulti_AgentFinancialResearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AutoGen Financial Research Multi-Agent Pipeline

**What is AutoGen?**

AutoGen is an open-source framework developed by Microsoft that helps developers build AI agents that can collaborate with each other to solve complex tasks. Instead of using a single Large Language Model (LLM), AutoGen lets you create multiple specialized agents that communicate, delegate work, use tools, execute code, and work together toward a goal.

**A five-agent AutoGen system that plans, researches, codes, executes, and writes a financial research report.**

Agents coordinate via GroupChat with a manager supervising the conversation rounds.

Requires: pip install pyautogen yfinance newsapi-python | OpenAI API key and News API key in Colab secrets

In [2]:
! pip install pyautogen yfinance newsapi-python

In [3]:
import autogen
print(f"AutoGen version: {autogen.__version__}")

ModuleNotFoundError: No module named 'autogen'

In [17]:
import autogen
from google.colab import userdata

llm_config = {
    "config_list": [{"model": "gpt-4-turbo", "api_key": userdata.get("OPENAI_API_KEY")}]
}
task = "Write a blog post about NVIDIA stock price performance in the past month. Today's date is 2026-07-05."

admin = autogen.UserProxyAgent(
    name="Admin",
    system_message="Give the task and send instructions to Writer to refine the blog post.",
    human_input_mode="ALWAYS",
    llm_config=llm_config
)
planner = autogen.AssistantAgent(
    name="Planner",
    system_message="Determine what information is needed to complete this task. All info must be retrieved using Python code. After each step is done by others, check progress. Instruct remaining steps. If a step fails, work around it.",
    llm_config=llm_config
)
engineer = autogen.AssistantAgent(
    name="Engineer",
    system_message="Write Python code to retrieve the data per the planner's plan. Use yfinance for stock data and NewsApiClient for news.",
    llm_config=llm_config
)
executor = autogen.UserProxyAgent(
    name="Executor",
    system_message="Execute the code written by the Engineer. Share the results.",
    human_input_mode="NEVER",
    code_execution_config={"work_dir": "coding", "use_docker": False}
)
writer = autogen.AssistantAgent(
    name="Writer",
    system_message="Write the financial blog post from the plan and data provided. Refine based on Admin feedback.",
    llm_config=llm_config
)

groupchat = autogen.GroupChat(
    agents=[admin, planner, engineer, executor, writer],
    messages=[],
    max_round=10
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)
admin.initiate_chat(manager, message=task)

ModuleNotFoundError: No module named 'autogen'